# Notebook 06 — Bases de Datos Vectoriales y Chunking

**Semana:** 6  
**Bloques:** 4.2 Bases de datos vectoriales · 4.3 Chunking e indexado  
**Tiempo estimado:** 3-4 horas

## Qué vas a construir

- Ingesta de un PDF real con tres estrategias de chunking distintas
- Indexado en Chroma con metadatos completos
- Comparativa visual: qué chunks recupera cada estrategia para las mismas queries
- Búsqueda con filtros por metadatos
- **Reto final:** indexa tres PDFs distintos en Chroma, busca en todos a la vez y filtra por fuente

## Prerrequisitos

```bash
pip install chromadb langchain langchain-text-splitters pymupdf openai python-dotenv
```

> **PDF para las prácticas:** puedes usar cualquier PDF en español que tengas a mano.
> Si no tienes ninguno, las celdas incluyen una alternativa que descarga
> el texto de una Wikipedia en lugar de un PDF.

---
## Celda 0 — Setup

In [ ]:
import os
import re
import json
import hashlib
import datetime
import numpy as np
import requests
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

load_dotenv()
OPENAI_KEY     = os.getenv("OPENAI_API_KEY")
cliente_openai = OpenAI(api_key=OPENAI_KEY)

# ── Función de embeddings reutilizada de la semana anterior ──────────────────
def generar_embeddings_batch(textos: list[str]) -> np.ndarray:
    respuesta = cliente_openai.embeddings.create(
        model="text-embedding-3-small",
        input=textos
    )
    vectores = [item.embedding for item in sorted(respuesta.data, key=lambda x: x.index)]
    return np.array(vectores)

def generar_embedding(texto: str) -> list[float]:
    r = cliente_openai.embeddings.create(model="text-embedding-3-small", input=texto)
    return r.data[0].embedding

# ── Directorios de trabajo ───────────────────────────────────────────────────
Path("datos_pdf").mkdir(exist_ok=True)
Path("chroma_db").mkdir(exist_ok=True)

# ── Cliente Chroma persistente ───────────────────────────────────────────────
chroma_cliente = chromadb.PersistentClient(path="./chroma_db")
print("Chroma inicializado. Colecciones existentes:",
      [c.name for c in chroma_cliente.list_collections()])

---
## Celda 1 — Carga y limpieza de texto desde PDF o Wikipedia

Carga tu propio PDF o usa la alternativa de Wikipedia para seguir sin PDF.

In [ ]:
# ── Opción A: carga un PDF propio ────────────────────────────────────────────

def cargar_pdf(ruta_pdf: str) -> list[dict]:
    """
    Extrae texto de un PDF página a página con PyMuPDF.
    Devuelve lista de dicts: {texto, pagina, fuente}
    """
    import fitz   # PyMuPDF
    doc     = fitz.open(ruta_pdf)
    paginas = []
    for i, pagina in enumerate(doc):
        texto = pagina.get_text("text").strip()
        if texto:   # ignora páginas vacías
            paginas.append({
                "texto":  texto,
                "pagina": i + 1,
                "fuente": Path(ruta_pdf).name
            })
    doc.close()
    return paginas


# ── Opción B: descarga artículo de Wikipedia (sin PDF) ───────────────────────

def cargar_wikipedia(titulo: str, idioma: str = "es") -> list[dict]:
    """
    Descarga el texto completo de un artículo de Wikipedia.
    Simula páginas dividiéndolo en bloques de ~1500 caracteres.
    """
    url  = f"https://{idioma}.wikipedia.org/api/rest_v1/page/summary/{requests.utils.quote(titulo)}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    extracto = resp.json().get("extract", "")

    # Para más contenido, usa la API de secciones
    url_secciones = f"https://{idioma}.wikipedia.org/w/api.php"
    params = {
        "action": "query", "format": "json",
        "titles": titulo,  "prop": "extracts",
        "explaintext": True, "exsectionformat": "plain"
    }
    resp2  = requests.get(url_secciones, params=params, timeout=10)
    paginas_wiki = resp2.json().get("query", {}).get("pages", {})
    texto_completo = list(paginas_wiki.values())[0].get("extract", extracto)

    # Divide en bloques simulando páginas
    tamano_bloque = 1500
    bloques = []
    for i in range(0, len(texto_completo), tamano_bloque):
        bloque = texto_completo[i:i + tamano_bloque].strip()
        if bloque:
            bloques.append({
                "texto":  bloque,
                "pagina": i // tamano_bloque + 1,
                "fuente": f"wikipedia_{titulo.replace(' ', '_')}.txt"
            })
    return bloques


# ── Limpieza básica de texto ──────────────────────────────────────────────────

def limpiar_texto(texto: str) -> str:
    """
    Limpieza básica: normaliza espacios, elimina líneas vacías múltiples,
    elimina caracteres de control.
    """
    texto = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", texto)  # control chars
    texto = re.sub(r" +",  " ",  texto)    # espacios múltiples
    texto = re.sub(r"\n{3,}", "\n\n", texto)  # más de 2 saltos de línea
    return texto.strip()


# ── Carga el documento ───────────────────────────────────────────────────────
# OPCIÓN A: pon la ruta a tu PDF
# RUTA_PDF = "datos_pdf/tu_documento.pdf"
# paginas = cargar_pdf(RUTA_PDF)

# OPCIÓN B: usa Wikipedia (descomenta si no tienes PDF)
TITULO_WIKIPEDIA = "Inteligencia artificial"
paginas = cargar_wikipedia(TITULO_WIKIPEDIA)

# Limpia el texto de cada página
for p in paginas:
    p["texto"] = limpiar_texto(p["texto"])

# Texto completo concatenado (para algunas estrategias de chunking)
texto_completo = "\n\n".join(p["texto"] for p in paginas)
fuente_nombre  = paginas[0]["fuente"] if paginas else "documento.txt"

print(f"Fuente:         {fuente_nombre}")
print(f"Páginas/bloques: {len(paginas)}")
print(f"Total caracteres: {len(texto_completo):,}")
print(f"Total palabras:   {len(texto_completo.split()):,}")
print(f"\nPrimeros 300 caracteres:\n{texto_completo[:300]}")

---
## Celda 2 — Las tres estrategias de chunking comparadas

Aplica las tres estrategias al mismo texto y observa:
- cuántos chunks produce cada una
- si los chunks respetan los límites naturales del texto
- el tamaño medio de cada chunk

In [ ]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

# ── Estrategia 1: tamaño fijo con solapamiento ────────────────────────────────

def chunking_fijo(
    texto: str,
    tamano: int = 500,
    solapamiento: int = 100
) -> list[str]:
    """Divide el texto cada `tamano` caracteres con `solapamiento` de contexto."""
    chunks = []
    inicio = 0
    while inicio < len(texto):
        fin = min(inicio + tamano, len(texto))
        chunks.append(texto[inicio:fin])
        if fin == len(texto):
            break
        inicio += tamano - solapamiento
    return chunks


# ── Estrategia 2: recursivo por separadores (LangChain) ───────────────────────

splitter_recursivo = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
    length_function=len,
)


# ── Estrategia 3: chunking semántico ─────────────────────────────────────────
# Nota: esta estrategia llama a la API de embeddings por cada oración,
# por lo que es más lenta y costosa. Aquí usamos una versión simplificada
# que agrupa párrafos por similitud en lugar de oraciones.

def chunking_semantico_parrafos(
    texto: str,
    umbral_corte: float = 0.65,
    max_chars: int = 800
) -> list[str]:
    """
    Agrupa párrafos consecutivos que hablan del mismo tema.
    Corta cuando la similitud entre párrafos consecutivos cae bajo el umbral.
    """
    parrafos = [p.strip() for p in texto.split("\n\n") if len(p.strip()) > 50]
    if len(parrafos) <= 1:
        return parrafos

    print(f"  Generando embeddings para {len(parrafos)} párrafos...")
    embeddings_parrafos = generar_embeddings_batch(parrafos)

    chunks     = []
    chunk_actual = parrafos[0]

    for i in range(1, len(parrafos)):
        # Similitud entre el párrafo actual y el anterior
        v_actual   = embeddings_parrafos[i]
        v_anterior = embeddings_parrafos[i - 1]
        norma_a    = np.linalg.norm(v_actual)
        norma_b    = np.linalg.norm(v_anterior)
        sim = float(np.dot(v_actual, v_anterior) / (norma_a * norma_b))

        chunk_potencial = chunk_actual + "\n\n" + parrafos[i]
        if sim < umbral_corte or len(chunk_potencial) > max_chars:
            chunks.append(chunk_actual)
            chunk_actual = parrafos[i]
        else:
            chunk_actual = chunk_potencial

    if chunk_actual:
        chunks.append(chunk_actual)

    return chunks


# ── Aplica las tres estrategias ───────────────────────────────────────────────

print("Aplicando estrategia 1: tamaño fijo...")
chunks_fijo      = chunking_fijo(texto_completo, tamano=500, solapamiento=100)

print("Aplicando estrategia 2: recursivo...")
chunks_recursivo = splitter_recursivo.split_text(texto_completo)

print("Aplicando estrategia 3: semántico...")
chunks_semantico = chunking_semantico_parrafos(texto_completo, umbral_corte=0.65)


# ── Estadísticas comparativas ─────────────────────────────────────────────────

def estadisticas_chunks(nombre: str, chunks: list[str]) -> None:
    tamanos = [len(c) for c in chunks]
    print(f"\n{'─'*50}")
    print(f"Estrategia: {nombre}")
    print(f"  N.º chunks:     {len(chunks)}")
    print(f"  Tamaño medio:   {np.mean(tamanos):.0f} chars")
    print(f"  Tamaño mín/máx: {min(tamanos)} / {max(tamanos)} chars")
    print(f"  Desv. estándar: {np.std(tamanos):.0f} chars")
    # Muestra el primer chunk como ejemplo
    print(f"  Primer chunk:   {chunks[0][:120].replace(chr(10), ' ')}...")

estadisticas_chunks("Fijo (500/100)",     chunks_fijo)
estadisticas_chunks("Recursivo (500/100)", chunks_recursivo)
estadisticas_chunks("Semántico",           chunks_semantico)


# ── Inspección visual: ¿dónde corta cada estrategia? ─────────────────────────
print("\n" + "="*60)
print("INSPECCIÓN: ¿Cortan en lugares lógicos?")
print("\n[Fijo] Fin del chunk 0 / inicio del chunk 1:")
print(f"  ...{chunks_fijo[0][-80:]}")
print(f"  {chunks_fijo[1][:80]}...")

print("\n[Recursivo] Fin del chunk 0 / inicio del chunk 1:")
print(f"  ...{chunks_recursivo[0][-80:]}")
print(f"  {chunks_recursivo[1][:80]}...")

print("\n¿Cuál respeta mejor los límites naturales del texto? → ...")

**Observaciones** (edita esta celda):

- Estrategia que produce chunks más uniformes en tamaño: ...
- Estrategia que respeta mejor los límites del texto: ...
- ¿El chunking semántico agrupó los párrafos de forma coherente?
- Para este documento concreto, ¿qué estrategia elegiríais en producción y por qué?

---
## Celda 3 — Indexado en Chroma con metadatos

Crea tres colecciones (una por estrategia) e indexa los chunks con metadatos completos.
Observa cómo añadir metadatos enriquece cada chunk sin aumentar el coste de embedding.

In [ ]:
def construir_metadatos(
    chunk: str,
    chunk_index: int,
    total_chunks: int,
    fuente: str,
    estrategia: str,
    pagina_aproximada: int | None = None
) -> dict:
    """Construye el dict de metadatos para un chunk."""
    return {
        "fuente":          fuente,
        "estrategia":      estrategia,
        "chunk_index":     chunk_index,
        "total_chunks":    total_chunks,
        "pagina":          pagina_aproximada or -1,
        "n_chars":         len(chunk),
        "n_palabras":      len(chunk.split()),
        "fecha_ingesta":   datetime.date.today().isoformat(),
        "es_primer_chunk": chunk_index == 0,
        "es_ultimo_chunk": chunk_index == total_chunks - 1,
    }


def indexar_chunks(
    nombre_coleccion: str,
    chunks: list[str],
    fuente: str,
    estrategia: str,
    batch_size: int = 50
) -> chromadb.Collection:
    """
    Indexa una lista de chunks en Chroma.
    Genera embeddings en batch y añade metadatos a cada chunk.
    Usa upsert para ser idempotente (seguro de ejecutar varias veces).
    """
    # Borra la colección si ya existe para empezar limpio
    try:
        chroma_cliente.delete_collection(nombre_coleccion)
    except Exception:
        pass

    coleccion = chroma_cliente.create_collection(
        name=nombre_coleccion,
        metadata={"hnsw:space": "cosine"}
    )

    total = len(chunks)
    print(f"Indexando {total} chunks en '{nombre_coleccion}'...")

    for inicio in range(0, total, batch_size):
        fin       = min(inicio + batch_size, total)
        lote      = chunks[inicio:fin]

        # IDs únicos basados en el hash del contenido + índice
        ids = [
            f"{estrategia}_{hashlib.md5(c.encode()).hexdigest()[:8]}_{i}"
            for i, c in enumerate(lote, inicio)
        ]

        # Embeddings en batch
        embeddings = generar_embeddings_batch(lote).tolist()

        # Metadatos
        metadatos = [
            construir_metadatos(c, i, total, fuente, estrategia)
            for i, c in enumerate(lote, inicio)
        ]

        coleccion.upsert(
            ids=ids,
            documents=lote,
            embeddings=embeddings,
            metadatas=metadatos
        )

        print(f"  Lote {inicio//batch_size + 1}: {fin}/{total} chunks indexados")

    print(f"✓ Colección '{nombre_coleccion}': {coleccion.count()} documentos")
    return coleccion


# ── Indexa las tres estrategias ───────────────────────────────────────────────
col_fijo      = indexar_chunks("chunks_fijo",      chunks_fijo,      fuente_nombre, "fijo")
col_recursivo = indexar_chunks("chunks_recursivo", chunks_recursivo, fuente_nombre, "recursivo")
col_semantico = indexar_chunks("chunks_semantico", chunks_semantico, fuente_nombre, "semantico")

print("\nColecciones disponibles en Chroma:")
for c in chroma_cliente.list_collections():
    col = chroma_cliente.get_collection(c.name)
    print(f"  {c.name}: {col.count()} documentos")

---
## Celda 4 — Comparativa de retrieval: ¿qué chunks recupera cada estrategia?

La prueba real: para las mismas queries, ¿qué fragmentos recupera cada estrategia?
¿Son igualmente relevantes? ¿Cuál da más contexto útil?

In [ ]:
def buscar_en_coleccion(
    coleccion: chromadb.Collection,
    query: str,
    n_results: int = 3,
    filtro_metadatos: dict | None = None
) -> list[dict]:
    """
    Busca en una colección Chroma y devuelve los resultados normalizados.
    Chroma devuelve distancia, no similitud — convertimos: similitud = 1 - distancia.
    """
    vector_query = generar_embedding(query)

    kwargs = {
        "query_embeddings": [vector_query],
        "n_results":        min(n_results, coleccion.count()),
        "include":          ["documents", "distances", "metadatas"]
    }
    if filtro_metadatos:
        kwargs["where"] = filtro_metadatos

    resp = coleccion.query(**kwargs)

    resultados = []
    for doc, dist, meta in zip(
        resp["documents"][0],
        resp["distances"][0],
        resp["metadatas"][0]
    ):
        resultados.append({
            "texto":      doc,
            "similitud":  round(1 - dist, 4),   # Chroma cosine: distance = 1 - similarity
            "metadatos":  meta
        })
    return resultados


def comparar_estrategias(query: str, n_results: int = 2) -> None:
    """Muestra los resultados de las tres estrategias para la misma query."""
    print(f"\n{'='*70}")
    print(f"QUERY: '{query}'")

    for nombre, coleccion in [
        ("FIJO",      col_fijo),
        ("RECURSIVO", col_recursivo),
        ("SEMÁNTICO", col_semantico),
    ]:
        resultados = buscar_en_coleccion(coleccion, query, n_results)
        print(f"\n  [{nombre}]")
        for i, r in enumerate(resultados, 1):
            meta  = r["metadatos"]
            texto = r["texto"].replace("\n", " ")[:120]
            print(f"    {i}. score={r['similitud']:.4f} | "
                  f"chunk {meta['chunk_index']}/{meta['total_chunks']} | "
                  f"{meta['n_chars']} chars")
            print(f"       {texto}...")


# ── Queries de comparativa ────────────────────────────────────────────────────
# Ajusta las queries según el documento que hayas cargado
QUERIES_COMPARATIVA = [
    "¿Cuáles son las principales aplicaciones de la inteligencia artificial?",
    "historia y origen del aprendizaje automático",
    "riesgos y problemas éticos de la IA",
    "redes neuronales profundas deep learning",
]

for query in QUERIES_COMPARATIVA:
    comparar_estrategias(query)

**Observaciones** (edita esta celda):

- Query en la que la estrategia fija es claramente inferior: ...
- ¿Hay queries donde el chunking semántico recupera chunks más coherentes? ¿Cuáles?
- ¿Qué score mínimo te parece un umbral razonable para este corpus?
- Si tuvieses que elegir una estrategia para producción con este documento, ¿cuál y por qué?

---
## Celda 5 — Búsqueda con filtros por metadatos

Los metadatos no son decorativos: permiten acotar la búsqueda antes de calcular similitudes.
Prueba los filtros y observa cómo cambia el comportamiento.

In [ ]:
# ── Experimento 1: filtrar por posición en el documento ───────────────────────
# Busca solo en los primeros chunks (introducción) o en los últimos (conclusión)

QUERY_FILTRO = "conclusiones y perspectivas futuras"
print(f"Query: '{QUERY_FILTRO}'\n")

# Sin filtro
sin_filtro = buscar_en_coleccion(col_recursivo, QUERY_FILTRO, n_results=2)
print("Sin filtro:")
for r in sin_filtro:
    print(f"  chunk {r['metadatos']['chunk_index']} | score={r['similitud']:.4f} | "
          f"{r['texto'][:80].replace(chr(10), ' ')}...")

# Filtrando solo los últimos chunks (índice > 75% del total)
total_chunks_col = col_recursivo.count()
indice_umbral    = int(total_chunks_col * 0.75)

# Nota: Chroma soporta operadores $gte, $lte, $eq, $ne, $in, $nin
con_filtro = buscar_en_coleccion(
    col_recursivo,
    QUERY_FILTRO,
    n_results=2,
    filtro_metadatos={"chunk_index": {"$gte": indice_umbral}}
)
print(f"\nSolo últimos chunks (índice >= {indice_umbral}):")
for r in con_filtro:
    print(f"  chunk {r['metadatos']['chunk_index']} | score={r['similitud']:.4f} | "
          f"{r['texto'][:80].replace(chr(10), ' ')}...")


# ── Experimento 2: filtrar por tamaño de chunk ────────────────────────────────
print("\n" + "─"*60)
print("Solo chunks sustanciales (> 300 caracteres):")

chunks_largos = buscar_en_coleccion(
    col_fijo,
    "aprendizaje profundo redes neuronales",
    n_results=3,
    filtro_metadatos={"n_chars": {"$gte": 300}}
)
for r in chunks_largos:
    m = r["metadatos"]
    print(f"  chunk {m['chunk_index']} | {m['n_chars']} chars | score={r['similitud']:.4f}")
    print(f"  {r['texto'][:100].replace(chr(10), ' ')}...")


# ── Experimento 3: inspección directa de metadatos ───────────────────────────
print("\n" + "─"*60)
print("Metadatos del primer chunk (colección recursiva):")
primer_chunk = col_recursivo.get(
    where={"chunk_index": {"$eq": 0}},
    include=["documents", "metadatas"]
)
if primer_chunk["ids"]:
    print(json.dumps(primer_chunk["metadatas"][0], indent=2, ensure_ascii=False))

---
## Reto final — Multi-documento: indexa tres fuentes y busca con filtros

Construye una colección unificada con tres documentos distintos.
Implementa un buscador que permita:
1. Buscar en todas las fuentes a la vez
2. Buscar solo en una fuente específica
3. Buscar en varias fuentes a la vez con un filtro `$in`

Lee el esqueleto y completa los `# TODO`.

In [ ]:
# ── Paso 1: descarga tres documentos de Wikipedia (o usa tus PDFs) ────────────

DOCUMENTOS = [
    {"titulo": "Machine learning",         "tipo": "tecnologia"},
    {"titulo": "Procesamiento del lenguaje natural", "tipo": "tecnologia"},
    {"titulo": "Ética de la inteligencia artificial", "tipo": "social"},
]

print("Cargando documentos...")
todos_los_chunks = []   # lista de dicts: {texto, metadatos}

for doc in DOCUMENTOS:
    print(f"  Cargando: {doc['titulo']}")
    paginas_doc = cargar_wikipedia(doc["titulo"])
    texto_doc   = "\n\n".join(limpiar_texto(p["texto"]) for p in paginas_doc)
    fuente_doc  = f"wikipedia_{doc['titulo'].replace(' ', '_')}.txt"

    chunks_doc = splitter_recursivo.split_text(texto_doc)
    for i, chunk in enumerate(chunks_doc):
        todos_los_chunks.append({
            "texto": chunk,
            "metadatos": {
                "fuente":        fuente_doc,
                "titulo":        doc["titulo"],
                "tipo":          doc["tipo"],
                "chunk_index":   i,
                "total_chunks":  len(chunks_doc),
                "n_chars":       len(chunk),
                "fecha_ingesta": datetime.date.today().isoformat(),
            }
        })
    print(f"    → {len(chunks_doc)} chunks")

print(f"\nTotal chunks multi-documento: {len(todos_los_chunks)}")


# ── Paso 2: indexa todos en una colección unificada ────────────────────────────

# TODO: crea una colección llamada "multi_doc" en Chroma
# e indexa todos los chunks de todos_los_chunks.
# Genera los embeddings en batch de 50 para no saturar la API.
# Cada documento debe tener un ID único — puedes usar f"{fuente}_{chunk_index}".

def indexar_multidoc(chunks_con_meta: list[dict], nombre_col: str = "multi_doc") -> chromadb.Collection:
    """Indexa una lista de {texto, metadatos} en Chroma."""
    try:
        chroma_cliente.delete_collection(nombre_col)
    except Exception:
        pass

    coleccion = chroma_cliente.create_collection(
        name=nombre_col,
        metadata={"hnsw:space": "cosine"}
    )

    # TODO: implementa el indexado en batches de 50
    # Tip: itera de 0 a len(chunks_con_meta) con paso 50
    #      en cada iteración, extrae textos, genera embeddings,
    #      construye IDs únicos y llama a coleccion.upsert()

    raise NotImplementedError("Implementa el indexado")


# col_multidoc = indexar_multidoc(todos_los_chunks)


# ── Paso 3: implementa el buscador multi-fuente ────────────────────────────────

def buscar_multidoc(
    query: str,
    coleccion: chromadb.Collection,
    fuentes: list[str] | None = None,   # None = busca en todas
    tipo: str | None = None,            # filtra por tipo de documento
    n_results: int = 5
) -> list[dict]:
    """
    Buscador con filtros opcionales por fuente y tipo.

    Ejemplos de uso:
    - buscar_multidoc("deep learning")                    # todas las fuentes
    - buscar_multidoc("deep learning", fuentes=["ML"])    # solo una fuente
    - buscar_multidoc("deep learning", tipo="tecnologia") # por tipo
    """
    # TODO: construye el filtro de metadatos según los parámetros
    # Tip: si fuentes tiene más de un elemento, usa {"fuente": {"$in": fuentes}}
    #      si fuentes tiene un elemento, usa {"fuente": {"$eq": fuentes[0]}}
    #      combina filtros con {"$and": [filtro1, filtro2]}
    raise NotImplementedError("Implementa el buscador")


# ── Paso 4: prueba las tres modalidades de búsqueda ────────────────────────────
# Descomenta cuando hayas implementado las funciones anteriores

# QUERY = "regulación ética y sesgos en algoritmos de IA"

# print("=== Búsqueda en TODAS las fuentes ===")
# for r in buscar_multidoc(QUERY, col_multidoc, n_results=4):
#     print(f"  [{r['metadatos']['titulo'][:30]}] {r['similitud']:.4f} | {r['texto'][:70]}...")

# print("\n=== Solo fuente: Ética de la IA ===")
# fuente_etica = "wikipedia_Ética_de_la_inteligencia_artificial.txt"
# for r in buscar_multidoc(QUERY, col_multidoc, fuentes=[fuente_etica], n_results=3):
#     print(f"  {r['similitud']:.4f} | {r['texto'][:80]}...")

# print("\n=== Solo tipo: tecnologia ===")
# for r in buscar_multidoc(QUERY, col_multidoc, tipo="tecnologia", n_results=3):
#     print(f"  [{r['metadatos']['titulo'][:30]}] {r['similitud']:.4f} | {r['texto'][:70]}...")

### Extensiones opcionales

1. **Qdrant en lugar de Chroma:** reimplementa la celda 3 usando Qdrant en lugar de Chroma. Compara la sintaxis de filtros. ¿Cuál encuentras más expresiva?

2. **Reindexado incremental:** escribe una función `actualizar_fuente(ruta_pdf, coleccion)` que elimine todos los chunks de esa fuente y los reindexe con el nuevo contenido, sin tocar el resto.

3. **Estadísticas del índice:** escribe una función que muestre un resumen del índice: cuántos chunks hay por fuente, cuántos por fecha de ingesta, y cuál es la distribución de tamaños.

In [ ]:
# Espacio para extensiones opcionales

---
## Autoevaluación de la semana

1. **¿Qué funcionó bien?**  
   ...

2. **¿Qué no entendiste del todo?**  
   ...

3. **¿Qué cambiarías la semana que viene?**  
   ...

4. **Tiempo real dedicado:**  
   ...

---
*Notebook 06 — Semana 6 del Plan de Formación AI Engineering*